In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

: 

In [ ]:
df = pd.read_csv('/content/customer_segments.csv')
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns)

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=['Customer ID'])

df = df[~df['Invoice'].astype(str).str.startswith('C')]

df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
print("Shape after cleaning:", df.shape)
df.head()

In [ ]:
df['Amount'] = df['Quantity'] * df['Price']

df[['Quantity', 'Price', 'Amount']].head()

In [ ]:
date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Amount', 'sum')
)



In [ ]:
print(rfm.shape)
rfm.describe()

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(rfm)

print(X[:5])

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

rfm['KMeans'] = kmeans.fit_predict(X)



In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)

rfm['DBSCAN'] = dbscan.fit_predict(X)

rfm['DBSCAN'].value_counts()

In [ ]:
pca = PCA(n_components=2)

pca_data = pca.fit_transform(X)

print(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:,0], pca_data[:,1], c=rfm['KMeans'])

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Customer Segments using PCA")

plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42)

tsne_data = tsne.fit_transform(X)

In [ ]:
plt.scatter(tsne_data[:,0], tsne_data[:,1], c=rfm['KMeans'])

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("Customer Segments using t-SNE")

plt.show()

In [ ]:
summary = rfm.groupby('KMeans')[['Recency', 'Frequency', 'Monetary']].mean()
summary